# Tokenizers

En esta sesión de Colab exploramos el mundo de los Tokenizers.

Puedes ejecutar este notebook en una CPU gratuita, o en tu propia máquina si lo prefieres.

## Recordatorio: 2 consejos importantes para usar Colab:

**Consejo 1:**

¡No te preocupes por las advertencias y los mensajes!

**Consejo 2:**

En medio de la ejecución de un Colab, puede aparecer un error como este:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

¡Es un mensaje de error súper engañoso! Por favor, no intentes cambiar las versiones de los paquetes...

Esto pasa porque Google cambió tu runtime de Colab, quizá porque Google Colab estaba muy ocupado. La solución es:

1. Menú Kernel >> Disconnect and delete runtime
2. Recarga el colab desde cero y ve al menú Edit >> Clear All Outputs
3. Conéctate a una nueva T4 con el botón de arriba a la derecha
4. Selecciona "View resources" en el menú de arriba a la derecha para confirmar que tienes GPU
5. Vuelve a ejecutar las celdas del colab, de arriba hacia abajo, empezando por las instalaciones de pip

Y todo debería funcionar muy bien. Si no, ¡pregúntame!

In [ ]:
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer

# Iniciar sesión en Hugging Face

1. Si todavía no lo has hecho, crea una cuenta GRATUITA de HuggingFace en https://huggingface.co y ve a Settings, luego Create a new API token, dándote permisos de escritura (write).

**IMPORTANTE:** al crear tu clave de API de Hugging Face, asegúrate de seleccionar permisos de lectura y escritura haciendo clic en la pestaña WRITE; si no, podrías tener problemas más adelante.

2. Pulsa el ícono de "llave" en el panel lateral izquierdo y agrega un nuevo secreto:
`HF_TOKEN = tu_token`

3. Ejecuta la celda de abajo para iniciar sesión.

In [ ]:
# Iniciar sesión en Hugging Face

hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("La clave de HF se ve bien por ahora")
else:
  print("La clave de HF no está configurada: pulsa la llave en la barra lateral izquierda")
login(hf_token, add_to_git_credential=True)

# Revisar la GPU de Google Colab

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('No conectado a una GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Éxito - Conectado a una T4")
  else:
    print("NO CONECTADO A UNA T4")

# Acceder a Llama 3.1 de Meta

Para usar el fantástico Llama 3.1, Meta exige que aceptes sus términos de servicio.

Visita la página de instrucciones del modelo en Hugging Face:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

En la parte superior de la página hay instrucciones para aceptar sus términos. Si es posible, usa el mismo correo que tu cuenta de Hugging Face.

En mi experiencia, la aprobación llega en un par de minutos. Una vez aprobado para cualquier modelo 3.1, aplica a toda la familia 3.1. Por alguna razón, a veces Meta no aprueba el acceso. Si te pasa, sigue [este](https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8?usp=sharing) notebook de solución de problemas.

Si la siguiente celda te da un error, revisa:
1. ¿Has iniciado sesión en HuggingFace? Prueba a ejecutar `login()` para comprobar que tu clave funciona
2. ¿Configuraste tu clave de API con permisos completos de lectura y escritura?
3. Si visitas la página de Llama 3.1 con el enlace de arriba, ¿aparece cerca de la parte superior que tienes acceso al modelo?

También preparé este Colab de solución de problemas para diagnosticar cualquier problema de conectividad con HuggingFace:
https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8?usp=sharing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)

In [ ]:
text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
tokens

In [ ]:
character_count = len(text)
word_count = len(text.split(' '))
token_count = len(tokens)
print(f"Hay {character_count} caracteres, {word_count} palabras y {token_count} tokens")

In [ ]:
tokenizer.decode(tokens)

In [ ]:
tokenizer.batch_decode(tokens)

In [ ]:
# tokenizer.vocab
tokenizer.get_added_vocab()

In [ ]:
len(tokenizer.vocab)

# Variantes Instruct de los modelos

Muchos modelos tienen una variante entrenada para chats.  
Normalmente se identifican con la palabra "Instruct" al final.  
Han sido entrenados para recibir prompts con un formato particular que incluye los prompts de sistema, usuario y asistente.  

Hay un método utilitario, `apply_chat_template`, que convierte la lista de mensajes que ya conocemos en el prompt de entrada correcto para este modelo.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

In [ ]:

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

## El momento "¡ajá!" crucial

Durante 2.5 semanas te he dado la impresión de que los LLM podían recibir una lista de diccionarios de Python de alguna forma:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
```

¡Pero un LLM es solo un modelo de ciencia de datos que toma una secuencia de números y predice la probabilidad del siguiente número! ¡No puedes pasarle un montón de objetos de Python a un modelo estadístico!

### Y ahora tienes la pieza que faltaba del rompecabezas...

Los mensajes en formato OpenAI se convierten:

1. ...en una secuencia de palabras con etiquetas especiales que separan el prompt de Sistema, Usuario y Asistente
2. ...luego las palabras se descomponen en fragmentos: "tokens"
3. ...luego los tokens se reemplazan por IDs de token, y esta es la secuencia de entrada

> La entrada de un LLM es una secuencia de IDs de token. La salida es la distribución de probabilidad del siguiente ID de token que debe seguir a esta entrada.

¡Eso es todo!

# Probando modelos nuevos

Ahora trabajaremos con 3 modelos:

Phi4 de Microsoft  
DeepSeek 3.1 de DeepSeek AI  
QwenCoder 2.5 de Alibaba Cloud

In [ ]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

In [ ]:
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print("Llama:")
tokens = tokenizer.encode(text)
print(tokens)
print(tokenizer.batch_decode(tokens))
print("\nPhi 4:")
tokens = phi4_tokenizer.encode(text)
print(tokens)
print(phi4_tokenizer.batch_decode(tokens))


In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi 4:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print(tokenizer.encode(text))
print()
print(phi4_tokenizer.encode(text))
print()
print(deepseek_tokenizer.encode(text))

In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nDeepSeek:")
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)
code = """
def hello_world(person):
  print("Hello", person)
"""
tokens = qwen_tokenizer.encode(code)
for token in tokens:
  print(f"{token}={qwen_tokenizer.decode(token)}")